# Object Detection: The YOLO Family and the Detector Landscape

Object detection answers two questions at once: *what* is in the image, and *where* is it? This notebook covers how YOLO turned detection into a fast single-pass problem, where it sits relative to two-stage detectors and transformer-based detectors, and how to run a pretrained YOLOv8 model with just a few lines of code.

In [ ]:
# pip install ultralytics matplotlib Pillow requests  # uncomment if needed

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import numpy as np
import requests
import io
import os

print('Dependencies imported.')

## 1. What Object Detection Actually Does

A classifier looks at a whole image and answers "what is this?" An object detector goes further: it draws a bounding box around every object it finds and labels each one.

Every detection output has three components:
- **Bounding box:** the rectangle, usually represented as `(x_center, y_center, width, height)` or `(x_min, y_min, x_max, y_max)`.
- **Class label:** which category the object belongs to.
- **Confidence score:** how certain the model is about that detection.

Multiple objects of different classes can appear in one image, and the model must handle all of them at once. That is what makes detection harder than classification.

## 2. The YOLO Family

### The original idea (YOLOv1, 2016)

"You Only Look Once" was a radical simplification. Before YOLO, the dominant approach (Faster R-CNN) used two stages:
1. Generate candidate regions (region proposals)
2. Classify each region

YOLO eliminated both stages with a single idea: **divide the image into a grid, and for each grid cell predict bounding boxes and class probabilities in one forward pass.**

The tradeoff at launch: slightly lower accuracy than Faster R-CNN, but 45 frames per second vs. 7 fps. For real-time applications, that was a worthwhile deal.

### Why single-pass detection is fast

Two-stage detectors run the backbone network twice in the worst case, plus the overhead of sampling and resizing hundreds of region proposals. YOLO runs the backbone exactly once and reads off all predictions from the final feature map. This means inference time scales only with the backbone size, not with the number of objects in the scene.

### Evolution to modern YOLO

The YOLO architecture has been substantially revised over the years by different research groups. The key improvements each generation brought:

| Version | Key change |
|---------|------------|
| YOLOv2 (2017) | Anchor boxes, batch normalization, multi-scale training |
| YOLOv3 (2018) | Multi-scale feature pyramid, better small object detection |
| YOLOv4 (2020) | Bag-of-freebies training tricks, CSP backbone |
| YOLOv5 (2020) | PyTorch rewrite, easy-to-use API, model scaling |
| YOLOv8 (2023) | Anchor-free head, improved architecture, ultralytics library |
| YOLOv11 (2024) | Further efficiency improvements, stronger small model |

YOLOv8 is the practical default for new projects in 2024-2025. The `ultralytics` library makes it trivial to use.

### Speed vs. accuracy

YOLOv8 comes in several size variants. These numbers are approximate (COCO val2017):

| Model | Parameters | mAP50-95 | Speed (CPU, ms/img) |
|-------|-----------|----------|--------------------|
| YOLOv8n (nano) | 3.2M | 37.3 | ~80 ms |
| YOLOv8s (small) | 11.2M | 44.9 | ~120 ms |
| YOLOv8m (medium) | 25.9M | 50.2 | ~230 ms |
| YOLOv8l (large) | 43.7M | 52.9 | ~375 ms |
| YOLOv8x (extra large) | 68.2M | 53.9 | ~479 ms |

The nano model is designed for edge devices and real-time pipelines. The extra-large model trades speed for accuracy.

## 3. Two Other Approaches Worth Knowing

### Two-stage detectors: Faster R-CNN

Faster R-CNN (Ren et al., 2015) works in two explicit stages. First, a **Region Proposal Network (RPN)** scans the feature map and proposes candidate bounding boxes that might contain objects. Second, each proposal is cropped, resized to a fixed size, and passed through a small classification head that refines the box coordinates and predicts the class.

Two-stage detectors tend to be more accurate on small objects and dense scenes because the RPN can generate many high-quality proposals. But they are slower because the classification head runs once per proposal, and scenes with many objects generate many proposals.

When to use Faster R-CNN: when accuracy is more important than speed, or when your images contain many small, densely packed objects.

### Transformer-based detection: DETR

DETR (Carion et al., 2020) replaces the entire anchor/proposal machinery with a transformer encoder-decoder. The encoder processes the image features. The decoder takes a fixed set of learned "object queries" and attends to the image features to predict objects directly.

The key innovation: DETR formulates detection as a *set prediction* problem. It does not use Non-Maximum Suppression (NMS), the post-processing step that all anchor-based detectors need to remove duplicate detections. Instead, it directly outputs exactly N predictions (where N is the fixed query count) and uses a bipartite matching loss during training to match predictions to ground truth objects.

DETR is conceptually clean but was originally slow to train (300 epochs on COCO). Follow-up models (Deformable DETR, DINO, RT-DETR) improved training efficiency significantly and are now competitive with YOLO on speed.

## 4. Using YOLOv8: The ultralytics API

The `ultralytics` library wraps YOLOv8 with a high-level API. Loading a model and running inference takes two lines. The library downloads weights automatically on first use.

In [ ]:
from ultralytics import YOLO

# Load the nano model (3.2M params, fastest)
# The library downloads yolov8n.pt automatically if not cached
model = YOLO('yolov8n.pt')

print(f'Model loaded: {model.info()}')

### Running Inference: Detections with Class, Confidence, and Bounding Box

`model.predict()` (or simply calling `model(image)`) returns a list of `Results` objects. The `conf` parameter sets the minimum confidence threshold; detections below it are discarded before you see them.

In [ ]:
# pip install ultralytics matplotlib Pillow requests  # uncomment if needed

import time
import requests
import io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from ultralytics import YOLO

# Load YOLOv8 nano -- lightest and fastest variant
model_n = YOLO('yolov8n.pt')
print('YOLOv8n loaded.')

# Download a sample image
SAMPLE_URL = 'https://ultralytics.com/images/bus.jpg'
SAMPLE_PATH = '/tmp/bus.jpg'

try:
    resp = requests.get(SAMPLE_URL, timeout=15)
    with open(SAMPLE_PATH, 'wb') as f:
        f.write(resp.content)
    print(f'Downloaded sample image to {SAMPLE_PATH}')
except Exception as e:
    print(f'Download failed: {e}. Using a placeholder image.')
    img_placeholder = Image.new('RGB', (640, 480), color=(120, 120, 120))
    img_placeholder.save(SAMPLE_PATH)

# --- Run inference with conf=0.3 ---
results = model_n.predict(SAMPLE_PATH, conf=0.3, verbose=False)
r = results[0]

print(f'\nNumber of detections (conf >= 0.3): {len(r.boxes)}')
print()
print(f'{"#":<4} {"Class":<15} {"Confidence":>12} {"x1":>7} {"y1":>7} {"x2":>7} {"y2":>7}')
print('-' * 65)

for i in range(len(r.boxes)):
    class_idx  = int(r.boxes.cls[i].item())
    class_name = model_n.names[class_idx]
    conf       = r.boxes.conf[i].item()
    x1, y1, x2, y2 = [round(v, 1) for v in r.boxes.xyxy[i].tolist()]
    print(f'{i:<4} {class_name:<15} {conf:>12.3f} {x1:>7} {y1:>7} {x2:>7} {y2:>7}')

### Visualization: Draw Bounding Boxes with Labels

We draw the bounding boxes manually using `matplotlib` so you can see exactly how the coordinates map to the image. The library's built-in `r.plot()` method is convenient but opaque; drawing manually makes the data structure concrete.

In [ ]:
def draw_detections(image_path, results_obj, model_names, conf_thresh=0.0, title='Detections'):
    """
    Draw bounding boxes and class labels on an image using matplotlib.

    Args:
        image_path:   path to the original image
        results_obj:  ultralytics Results object (results[0])
        model_names:  dict mapping class_id -> class_name (model.names)
        conf_thresh:  only draw detections above this confidence
        title:        plot title
    """
    img = Image.open(image_path).convert('RGB')
    fig, ax = plt.subplots(1, figsize=(10, 7))
    ax.imshow(img)

    cmap = plt.cm.get_cmap('tab20', 80)

    for i in range(len(results_obj.boxes)):
        conf = results_obj.boxes.conf[i].item()
        if conf < conf_thresh:
            continue

        class_idx  = int(results_obj.boxes.cls[i].item())
        class_name = model_names[class_idx]
        x1, y1, x2, y2 = results_obj.boxes.xyxy[i].tolist()

        color = cmap(class_idx % 20)
        rect  = mpatches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(
            x1, max(y1 - 6, 0),
            f'{class_name} {conf:.2f}',
            color='white', fontsize=9,
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85),
        )

    ax.set_title(title)
    ax.axis('off')
    plt.tight_layout()
    plt.show()


# Draw detections from YOLOv8n at conf=0.3
draw_detections(SAMPLE_PATH, r, model_n.names, conf_thresh=0.3,
                title=f'YOLOv8n detections (conf >= 0.3)  --  {len(r.boxes)} boxes')

## Model Size Comparison: nano, small, medium

YOLOv8 comes in five sizes. Here we load nano, small, and medium, count their parameters, and time inference on the same image so you can see the speed/accuracy trade-off concretely.

In [ ]:
import torch

model_configs = [
    ('yolov8n.pt', 'nano'),
    ('yolov8s.pt', 'small'),
    ('yolov8m.pt', 'medium'),
]

size_results = []

for weights, size_name in model_configs:
    m = YOLO(weights)

    # Parameter count
    n_params = sum(p.numel() for p in m.model.parameters())

    # Warm-up run (first run includes JIT tracing overhead)
    _ = m.predict(SAMPLE_PATH, conf=0.3, verbose=False)

    # Timed run (average of 3 runs)
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        res = m.predict(SAMPLE_PATH, conf=0.3, verbose=False)
        times.append(time.perf_counter() - t0)

    avg_ms    = np.mean(times) * 1000
    n_dets    = len(res[0].boxes)

    size_results.append({
        'variant':    size_name,
        'weights':    weights,
        'params_M':   n_params / 1e6,
        'latency_ms': avg_ms,
        'detections': n_dets,
    })

    print(f'YOLOv8-{size_name:<7}  params={n_params/1e6:.1f}M  '
          f'latency={avg_ms:.1f}ms  detections={n_dets}')

# --- Bar chart: latency vs. model size ---
variants  = [r['variant']    for r in size_results]
latencies = [r['latency_ms'] for r in size_results]
params    = [r['params_M']   for r in size_results]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(variants, latencies, color=['steelblue', 'darkorange', 'firebrick'])
axes[0].set_ylabel('Inference latency (ms)')
axes[0].set_title('YOLOv8 inference latency by size')
for i, v in enumerate(latencies):
    axes[0].text(i, v + 1, f'{v:.0f}ms', ha='center')

axes[1].bar(variants, params, color=['steelblue', 'darkorange', 'firebrick'])
axes[1].set_ylabel('Parameters (millions)')
axes[1].set_title('YOLOv8 parameter count by size')
for i, v in enumerate(params):
    axes[1].text(i, v + 0.2, f'{v:.1f}M', ha='center')

plt.tight_layout()
plt.savefig('/tmp/yolo_size_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## COCO Evaluation: mAP@50 and mAP@50-95

The standard evaluation metric for object detection is mean Average Precision (mAP). `mAP@50` measures precision at IoU threshold 0.5 (a box is "correct" if it overlaps the ground truth by at least 50%). `mAP@50-95` averages over IoU thresholds from 0.50 to 0.95 in steps of 0.05 and is a stricter measure of localization quality.

`model.val(data="coco.yaml")` runs COCO evaluation. It requires the COCO validation images to be downloaded locally. Below we show the call and what the output fields mean.

In [ ]:
# Full COCO val2017 evaluation requires ~1GB of images and takes ~10 min.
# Use coco128.yaml for a quick sanity check (128 images, included with ultralytics).

# Quick evaluation on COCO128 (downloads automatically)
try:
    eval_model = YOLO('yolov8n.pt')
    metrics = eval_model.val(data='coco128.yaml', verbose=False)

    print('YOLOv8n evaluation on COCO128:')
    print(f'  mAP@50:     {metrics.box.map50:.4f}')
    print(f'  mAP@50-95:  {metrics.box.map:.4f}')
    print(f'  Precision:  {metrics.box.mp:.4f}')
    print(f'  Recall:     {metrics.box.mr:.4f}')
    print()
    print('What these numbers mean:')
    print('  mAP@50    -- fraction of detected objects that overlap ground truth by >= 50%')
    print('  mAP@50-95 -- stricter; requires increasingly tight localization')
    print('  Precision -- of all predicted boxes, what fraction are correct')
    print('  Recall    -- of all ground-truth boxes, what fraction were detected')

except Exception as e:
    print(f'Evaluation failed or data not available: {e}')
    print()
    print('To run full COCO evaluation:')
    print("   metrics = model.val(data='coco.yaml')")
    print("   print(metrics.box.map50)     # mAP@50")
    print("   print(metrics.box.map)       # mAP@50-95")
    print()
    print('Published YOLOv8n numbers on COCO val2017:')
    print('   mAP@50:     0.527')
    print('   mAP@50-95:  0.373')

## Export to ONNX

ONNX (Open Neural Network Exchange) is a portable format for ML models. Exporting to ONNX lets you run the model with ONNX Runtime, TensorRT, or deploy it on edge hardware without a PyTorch dependency.

`model.export(format="onnx")` creates a `.onnx` file in the same directory as the weights.

In [ ]:
import os

export_model = YOLO('yolov8n.pt')

try:
    # Export to ONNX format
    onnx_path = export_model.export(format='onnx', dynamic=False, simplify=True)
    print(f'Exported to: {onnx_path}')

    # Verify the file exists and check its size
    if onnx_path and os.path.exists(onnx_path):
        size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
        print(f'File size: {size_mb:.1f} MB')
    else:
        # ultralytics may return the path as a Path object or a string
        import glob
        candidates = glob.glob('**/*.onnx', recursive=True) + glob.glob('/tmp/**/*.onnx', recursive=True)
        if candidates:
            print(f'ONNX file found at: {candidates[0]}')
        else:
            print('ONNX file location not found automatically; check the working directory.')

    print()
    print('You can now run inference with ONNX Runtime:')
    print("""
    import onnxruntime as ort
    import numpy as np

    session = ort.InferenceSession('yolov8n.onnx')
    input_name = session.get_inputs()[0].name
    dummy_input = np.random.randn(1, 3, 640, 640).astype(np.float32)
    outputs = session.run(None, {input_name: dummy_input})
    print('ONNX output shape:', outputs[0].shape)
    """)

except Exception as e:
    print(f'Export failed: {e}')
    print("Typical fix: pip install onnx onnxsim")

## Simulating Frame-by-Frame Video Processing

In a real video pipeline you would read frames from a video file with `cv2.VideoCapture`. Here we simulate that by running YOLOv8 on a list of images and measuring per-frame latency, which is exactly what a video loop does.

In [ ]:
# Simulate a 10-frame "video" by repeating the sample image
# In a real pipeline: cap = cv2.VideoCapture('video.mp4'); ret, frame = cap.read()

frame_paths = [SAMPLE_PATH] * 10   # 10 identical frames as a stand-in
video_model = YOLO('yolov8n.pt')

frame_latencies = []
frame_detections = []

print(f'{"Frame":<6} {"Latency (ms)":>14} {"Detections":>12}')
print('-' * 36)

for frame_idx, frame_path in enumerate(frame_paths):
    t0 = time.perf_counter()
    frame_results = video_model.predict(frame_path, conf=0.3, verbose=False)
    latency_ms = (time.perf_counter() - t0) * 1000

    n_det = len(frame_results[0].boxes)
    frame_latencies.append(latency_ms)
    frame_detections.append(n_det)

    print(f'{frame_idx:<6} {latency_ms:>14.1f} {n_det:>12}')

avg_latency = np.mean(frame_latencies)
fps_estimate = 1000.0 / avg_latency

print()
print(f'Average latency: {avg_latency:.1f} ms/frame')
print(f'Estimated throughput: {fps_estimate:.1f} FPS')
print()
print('Note: the first frame is slower (model initialization / JIT warmup).')
print('Frames 2-10 represent steady-state inference speed.')

# Plot per-frame latency
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(len(frame_latencies)), frame_latencies, color='steelblue')
ax.axhline(avg_latency, color='firebrick', linestyle='--', label=f'avg={avg_latency:.0f}ms')
ax.set_xlabel('Frame index')
ax.set_ylabel('Latency (ms)')
ax.set_title('YOLOv8n per-frame inference latency')
ax.legend()
plt.tight_layout()
plt.show()

## YOLOv8 vs. Faster R-CNN: Speed and Detections

Faster R-CNN is torchvision's reference two-stage detector. It is slower than YOLOv8 but was designed for accuracy. Running both on the same image lets you compare detection quality and speed directly.

`torchvision.models.detection.fasterrcnn_resnet50_fpn` outputs a list of dicts with `boxes`, `labels`, and `scores` keys.

In [ ]:
import torch
import torchvision
import torchvision.transforms as T
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn,
    FasterRCNN_ResNet50_FPN_Weights,
)

# COCO class names for Faster R-CNN (91 classes, 0=background)
COCO_NAMES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A', 'stop sign',
    'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack', 'umbrella', 'N/A',
    'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
    'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard',
    'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass', 'cup', 'fork',
    'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli',
    'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant',
    'bed', 'N/A', 'dining table', 'N/A', 'N/A', 'toilet', 'N/A', 'tv', 'laptop',
    'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster',
    'sink', 'refrigerator', 'N/A', 'book', 'clock', 'vase', 'scissors',
    'teddy bear', 'hair drier', 'toothbrush',
]

# Load Faster R-CNN
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
frcnn = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
frcnn.eval().to(device)
print(f'Faster R-CNN parameters: {sum(p.numel() for p in frcnn.parameters())/1e6:.1f}M')

# Preprocess the sample image for Faster R-CNN (expects a list of FloatTensors in [0,1])
img_pil  = Image.open(SAMPLE_PATH).convert('RGB')
img_tensor = T.ToTensor()(img_pil).to(device)

CONF_THRESH = 0.5

# --- Time Faster R-CNN ---
_ = frcnn([img_tensor])   # warmup
t0 = time.perf_counter()
with torch.no_grad():
    frcnn_output = frcnn([img_tensor])[0]
frcnn_ms = (time.perf_counter() - t0) * 1000

frcnn_dets = [
    {'class': COCO_NAMES[int(l)], 'conf': float(s), 'box': b.tolist()}
    for l, s, b in zip(frcnn_output['labels'], frcnn_output['scores'], frcnn_output['boxes'])
    if float(s) >= CONF_THRESH
]

# --- Time YOLOv8n ---
yolo_compare = YOLO('yolov8n.pt')
_ = yolo_compare.predict(SAMPLE_PATH, conf=CONF_THRESH, verbose=False)   # warmup
t0 = time.perf_counter()
yolo_result = yolo_compare.predict(SAMPLE_PATH, conf=CONF_THRESH, verbose=False)[0]
yolo_ms = (time.perf_counter() - t0) * 1000

yolo_dets = [
    {'class': yolo_compare.names[int(yolo_result.boxes.cls[i])],
     'conf':  float(yolo_result.boxes.conf[i])}
    for i in range(len(yolo_result.boxes))
]

# --- Print comparison ---
print(f'\n{"Model":<20}  {"Latency (ms)":>13}  {"Detections (conf>=0.5)":>24}')
print('-' * 62)
print(f'{"YOLOv8n":<20}  {yolo_ms:>13.1f}  {len(yolo_dets):>24}')
print(f'{"Faster R-CNN (R50)":<20}  {frcnn_ms:>13.1f}  {len(frcnn_dets):>24}')

print(f'\nYOLOv8n detections:')
for d in yolo_dets:
    print(f'  {d["class"]} ({d["conf"]:.3f})')

print(f'\nFaster R-CNN detections:')
for d in frcnn_dets:
    print(f'  {d["class"]} ({d["conf"]:.3f})')

# --- Side-by-side visualization ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
cmap = plt.cm.get_cmap('tab20', 80)

for ax, dets, model_name, use_yolo in [
    (axes[0], yolo_dets, 'YOLOv8n', True),
    (axes[1], frcnn_dets, 'Faster R-CNN (R50)', False),
]:
    ax.imshow(img_pil)
    if use_yolo:
        for i in range(len(yolo_result.boxes)):
            if float(yolo_result.boxes.conf[i]) < CONF_THRESH:
                continue
            cls   = int(yolo_result.boxes.cls[i].item())
            cname = yolo_compare.names[cls]
            conf  = float(yolo_result.boxes.conf[i])
            x1, y1, x2, y2 = yolo_result.boxes.xyxy[i].tolist()
            rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                       linewidth=2, edgecolor=cmap(cls % 20), facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, max(y1-6,0), f'{cname} {conf:.2f}', color='white', fontsize=7,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=cmap(cls%20), alpha=0.85))
    else:
        for j, (l, s, b) in enumerate(zip(frcnn_output['labels'],
                                           frcnn_output['scores'],
                                           frcnn_output['boxes'])):
            if float(s) < CONF_THRESH:
                continue
            cls = int(l)
            x1, y1, x2, y2 = b.tolist()
            rect = mpatches.Rectangle((x1, y1), x2-x1, y2-y1,
                                       linewidth=2, edgecolor=cmap(cls%20), facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, max(y1-6,0), f'{COCO_NAMES[cls]} {float(s):.2f}',
                    color='white', fontsize=7,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=cmap(cls%20), alpha=0.85))
    ax.set_title(f'{model_name}\n{len(dets)} detections  |  {yolo_ms if use_yolo else frcnn_ms:.0f} ms')
    ax.axis('off')

plt.tight_layout()
plt.savefig('/tmp/yolo_vs_frcnn.png', dpi=80, bbox_inches='tight')
plt.show()

In [ ]:
# Download a sample COCO-style image
# Using a Wikipedia commons image of a street scene with multiple objects
image_url = 'https://ultralytics.com/images/bus.jpg'

try:
    resp = requests.get(image_url, timeout=15)
    img = Image.open(io.BytesIO(resp.content)).convert('RGB')
    sample_path = '/tmp/sample_bus.jpg'
    img.save(sample_path)
    print(f'Downloaded image: {img.size} pixels')
    plt.figure(figsize=(8, 5))
    plt.imshow(img)
    plt.title('Input image')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Download failed: {e}')
    # Create a dummy test image if download fails
    img = Image.new('RGB', (640, 480), color=(100, 120, 140))
    sample_path = '/tmp/sample_bus.jpg'
    img.save(sample_path)
    print('Created placeholder image.')

In [ ]:
# Run inference on the image
results = model(sample_path)

# results is a list, one element per image
r = results[0]

print(f'Type of results[0]: {type(r)}')
print(f'Number of detections: {len(r.boxes)}')
print()

In [ ]:
# Parsing the Results object
# r.boxes contains all detected bounding boxes

boxes = r.boxes

print('Detection results:')
print(f'{"#":<4} {"Class":<15} {"Confidence":>12} {"Bounding box (xyxy)":>35}')
print('-' * 70)

for i in range(len(boxes)):
    # Class index and name
    class_idx  = int(boxes.cls[i].item())
    class_name = model.names[class_idx]

    # Confidence score
    conf = boxes.conf[i].item()

    # Bounding box in (x_min, y_min, x_max, y_max) format
    x1, y1, x2, y2 = boxes.xyxy[i].tolist()

    print(f'{i:<4} {class_name:<15} {conf:>12.3f} '
          f'({x1:6.1f}, {y1:6.1f}, {x2:6.1f}, {y2:6.1f})')

print()
print('Available box formats:')
print('  boxes.xyxy   -> [x_min, y_min, x_max, y_max]  (absolute pixels)')
print('  boxes.xywh   -> [x_center, y_center, width, height]  (absolute pixels)')
print('  boxes.xyxyn  -> normalized [0,1] version of xyxy')
print('  boxes.xywhn  -> normalized [0,1] version of xywh')

In [ ]:
# Draw detections manually so we understand what the library is doing
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: original image
axes[0].imshow(img)
axes[0].set_title('Original image')
axes[0].axis('off')

# Right: manually drawn detections
axes[1].imshow(img)

# Color map for different classes
colors = plt.cm.get_cmap('tab10', 10)
drawn_classes = {}

for i in range(len(boxes)):
    class_idx  = int(boxes.cls[i].item())
    class_name = model.names[class_idx]
    conf       = boxes.conf[i].item()
    x1, y1, x2, y2 = boxes.xyxy[i].tolist()

    color = colors(class_idx % 10)
    rect  = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                linewidth=2, edgecolor=color, facecolor='none')
    axes[1].add_patch(rect)
    axes[1].text(x1, y1 - 4, f'{class_name} {conf:.2f}',
                  color='white', fontsize=8,
                  bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.8))

axes[1].set_title('YOLOv8n detections (manually drawn)')
axes[1].axis('off')

plt.tight_layout()
plt.savefig('/tmp/yolo_detections.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# The library also has its own built-in visualization
annotated = r.plot()  # returns a numpy array (BGR)

# Convert BGR to RGB for matplotlib
annotated_rgb = annotated[:, :, ::-1]

plt.figure(figsize=(9, 6))
plt.imshow(annotated_rgb)
plt.title('YOLOv8n: built-in annotation (r.plot())')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# You can also pass a URL directly to the model
url = 'https://ultralytics.com/images/zidane.jpg'

try:
    results_url = model(url)
    r_url = results_url[0]

    print(f'Detections from URL image: {len(r_url.boxes)}')
    for i in range(len(r_url.boxes)):
        class_name = model.names[int(r_url.boxes.cls[i].item())]
        conf       = r_url.boxes.conf[i].item()
        print(f'  {class_name}: {conf:.3f}')

    annotated_url = r_url.plot()[:, :, ::-1]
    plt.figure(figsize=(9, 6))
    plt.imshow(annotated_url)
    plt.title('YOLOv8n on URL image')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

except Exception as e:
    print(f'URL inference skipped: {e}')

## 5. The Results Object in Depth

The `Results` object returned by `model(image)` is worth understanding fully before you build anything on top of it.

In [ ]:
r = results[0]  # use the bus image result

print('--- Results object attributes ---')
print(f'r.orig_img:  original image as numpy array, shape={r.orig_img.shape}')
print(f'r.orig_shape: original image size (H, W) = {r.orig_shape}')
print(f'r.names:     class name dict, e.g. r.names[0] = "{r.names[0]}"')
print()

print('--- Boxes tensor ---')
print(f'r.boxes.data:  raw tensor, shape={r.boxes.data.shape}')
print(f'  columns: [x1, y1, x2, y2, confidence, class_index]')
print()

print('--- Converting to Python list ---')
detections = []
for i in range(len(r.boxes)):
    det = {
        'class':  r.names[int(r.boxes.cls[i].item())],
        'conf':   round(r.boxes.conf[i].item(), 4),
        'box':    [round(v, 1) for v in r.boxes.xyxy[i].tolist()],
    }
    detections.append(det)

for d in detections:
    print(f'  {d}')

print()
print('--- Filtering by confidence ---')
confident_dets = [d for d in detections if d['conf'] >= 0.5]
print(f'{len(confident_dets)} detections with confidence >= 0.50:')
for d in confident_dets:
    print(f'  {d["class"]} ({d["conf"]:.3f})')

print()
print('--- Filtering by class ---')
person_dets = [d for d in detections if d['class'] == 'person']
print(f'{len(person_dets)} person detections')

## 6. Confidence Threshold and What It Means

Every detection comes with a confidence score between 0 and 1. This score combines the model's certainty that a box contains any object at all, with its certainty about the specific class. Higher is more confident.

The default confidence threshold in ultralytics is 0.25: detections below this are discarded before you see them. You can change it with the `conf` parameter.

In [ ]:
# Compare detection counts at different confidence thresholds
thresholds = [0.1, 0.25, 0.5, 0.7]

fig, axes = plt.subplots(1, len(thresholds), figsize=(18, 5))

for ax, conf_thresh in zip(axes, thresholds):
    r_thresh = model(sample_path, conf=conf_thresh)[0]
    annotated = r_thresh.plot()[:, :, ::-1]
    ax.imshow(annotated)
    ax.set_title(f'conf >= {conf_thresh}\n({len(r_thresh.boxes)} detections)')
    ax.axis('off')

plt.suptitle('Effect of confidence threshold on YOLOv8n output', y=1.01)
plt.tight_layout()
plt.savefig('/tmp/conf_threshold_comparison.png', dpi=80, bbox_inches='tight')
plt.show()

## Exercise

**Exercise 1: Class frequency**
Run `model` on 3 different images (use URLs or local files). For each image, extract the class names of all detections with confidence >= 0.3. Print a frequency count: how many times did each class appear across all three images?

**Exercise 2: IoU computation**
Intersection over Union (IoU) is the standard metric for evaluating bounding box overlap. Given two boxes in `(x1, y1, x2, y2)` format, write a function `iou(box1, box2)` that computes IoU. Test it on two overlapping boxes from your detections.

Expected: `iou([0,0,10,10], [5,5,15,15]) == 25/(100+100-25) == 0.1429`

**Exercise 3: Model size comparison**
Load `yolov8n.pt` and `yolov8s.pt`. Run both on the same image and compare:
- Number of detections at conf=0.25
- Which classes appear in one but not the other
- Inference time (use `time.time()` around the `model()` call)

**Exercise 4: Video-style tracking (static simulation)**
Download 3 images that simulate consecutive frames of a video (find a video thumbnail, a still, etc.). Run YOLO on each and print the class and confidence of the highest-confidence detection in each "frame". This simulates what a real-time video pipeline would process.

## Exercise: Run YOLOv8 on Your Own Image

Download or use any image you like, run YOLOv8n on it, filter detections above conf=0.5, and print a clean summary.

In [ ]:
# YOUR CODE HERE
#
# Steps:
# 1. Choose an image: download from a URL, or use a local file path.
#    Example download:
#      import requests, io
#      from PIL import Image
#      url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/.../320px-....jpg'
#      img = Image.open(io.BytesIO(requests.get(url, timeout=15).content)).convert('RGB')
#      img.save('/tmp/my_image.jpg')
#
# 2. Run YOLOv8n:
#      model = YOLO('yolov8n.pt')
#      results = model.predict('/tmp/my_image.jpg', conf=0.5, verbose=False)
#      r = results[0]
#
# 3. Print a summary table:
#      for i in range(len(r.boxes)):
#          class_name = model.names[int(r.boxes.cls[i])]
#          conf       = r.boxes.conf[i].item()
#          x1, y1, x2, y2 = r.boxes.xyxy[i].tolist()
#          print(f'  {class_name}: {conf:.3f}  bbox=({x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f})')
#
# 4. Print total detections and the highest-confidence class.

raise NotImplementedError